## Задача 1

Для кода с порождающей матрицей

$$
G = \left[\begin{array}{cccccc}
1 & 0 & 1 & 1 & 0 & 1\\
1 & 0 & 1 & 0 & 1 & 0\\
1 & 1 & 0 & 1 & 0 & 0
\end{array}\right]
$$

нужно:

- построить решетку по проверочной матрице;
- построить синдромную решетку;
- убедиться, что они совпадают с точностью до нумерации узлов.

Ниже это делается программно:

- из `G` строится проверочная матрица `H`;
- перечисляются все кодовые слова;
- строится синдромная решетка по частичным синдромам;
- отдельно строится минимальная решетка по множествам будущих;
- затем сравниваются множества будущих на каждом ярусе.


### Импорт


In [14]:
from itertools import product

### Вспомогательные функции

Здесь определены:

- операции над векторами по модулю 2;
- перечисление всех кодовых слов;
- построение проверочной матрицы `H` по `G`;
- форматированный вывод.


In [15]:
def xor_vectors(a, b):
    return tuple((x + y) % 2 for x, y in zip(a, b))  # складываем векторы по модулю 2


def dot(a, b):
    return (
        sum(x * y for x, y in zip(a, b)) % 2
    )  # считаем скалярное произведение векторов по модулю 2


def format_vector(vector):
    return "".join(map(str, vector))  # преобразуем вектор в строку


def print_matrix(name, matrix):  # выводим матрицу
    print(f"{name} =")
    for row in matrix:
        print(" ", row)


def rank_gf2(rows):  # считаем ранг матрицы над GF(2)
    matrix = [
        list(row) for row in rows if any(row)
    ]  # преобразуем строки в списки и убираем нулевые
    if not matrix:
        return 0

    row_count = len(matrix)
    col_count = len(matrix[0])
    rank = 0
    pivot_row = 0

    for col in range(col_count):  # перебираем все столбцы
        pivot = None
        for row in range(pivot_row, row_count):  # перебираем все строки
            if matrix[row][col] == 1:  # если в столбце есть 1
                pivot = row  # запоминаем строку
                break

        if pivot is None:
            continue  # если в столбце нет 1, то переходим к следующему столбцу

        matrix[pivot_row], matrix[pivot] = (
            matrix[pivot],
            matrix[pivot_row],
        )  # меняем местами строки

        for row in range(row_count):  # перебираем все строки
            if (
                row != pivot_row and matrix[row][col] == 1
            ):  # если в строке нет 1, то переходим к следующей строке
                matrix[row] = [
                    (matrix[row][j] + matrix[pivot_row][j]) % 2
                    for j in range(col_count)  # складываем векторы по модулю 2
                ]

        rank += 1  # увеличиваем ранг
        pivot_row += 1  # переходим к следующей строке

        if pivot_row == row_count:
            break  # если достигли последней строки, то выходим из цикла

    return rank


def all_codewords(generator_matrix):

    k = len(generator_matrix)
    n = len(generator_matrix[0])
    words = []

    for message in product([0, 1], repeat=k):  # перебираем все возможные слова
        codeword = []  # создаём пустой список для кодового слова
        for column in range(n):  # перебираем все столбцы
            value = (
                sum(message[row] * generator_matrix[row][column] for row in range(k))
                % 2
            )  # считаем значение для каждого столбца
            codeword.append(value)  # добавляем значение в кодовое слово
        words.append(tuple(codeword))  # добавляем кодовое слово в список

    return words


def parity_check_matrix_from_generator(
    generator_matrix,
):  # строим проверочную матрицу по порождающей
    n = len(generator_matrix[0])  # длина кодового слова
    orthogonal_vectors = []  # создаём пустой список для ортогональных векторов

    for vector in product([0, 1], repeat=n):  # перебираем все возможные векторы
        if all(
            dot(row, vector) == 0 for row in generator_matrix
        ):  # если вектор ортогонален всем строкам порождающей матрицы
            orthogonal_vectors.append(
                vector
            )  # добавляем вектор в список ортогональных векторов

    basis = []  # создаём пустой список для базиса
    for vector in orthogonal_vectors:  # перебираем все ортогональные векторы
        if not any(vector):  # если вектор нулевой, то переходим к следующему вектору
            continue  # если вектор нулевой, то переходим к следующему вектору
        candidate = basis + [vector]  # добавляем вектор в базис
        if rank_gf2(candidate) > rank_gf2(
            basis
        ):  # если ранг базиса больше, то добавляем вектор в базис
            basis.append(vector)  # добавляем вектор в базис

    return [list(row) for row in basis]  # преобразуем базис в список списков

### Построение двух решеток

1.  **Синдромная решетка**:
    узлы на ярусе `i` — частичные синдромы после первых `i` символов.

2.  **Минимальная решетка по множествам будущих**:
    два префикса относятся к одному узлу, если множества допустимых суффиксов
    у них одинаковы.

Если на каждом ярусе множества будущих совпадают, то решетки изоморфны
с точностью до нумерации узлов.


In [16]:
def partial_syndrome(
    prefix, columns, syndrome_length
):  # частичный синдром для префикса
    if not columns:  # если нет столбцов, то возвращаем нулевой вектор
        return (0,) * syndrome_length

    syndrome = (0,) * len(columns[0])  # создаём нулевой вектор
    for bit, column in zip(prefix, columns):  # перебираем все столбцы
        if bit == 1:  # если бит равен 1, то складываем векторы по модулю 2
            syndrome = xor_vectors(syndrome, column)  # складываем векторы по модулю 2
    return syndrome


def build_syndrome_trellis(codewords, parity_check_matrix):  # строим синдромную решетку
    n = len(codewords[0])  # длина кодового слова
    syndrome_length = len(parity_check_matrix)  # длина синдрома
    columns = list(zip(*parity_check_matrix))  # создаём список столбцов
    layers = []  # создаём список слоев
    future_sets_by_layer = []  # создаём список будущих множеств

    for layer in range(n + 1):  # перебираем все слои
        nodes = {}  # создаём словарь узлов
        for word in codewords:  # перебираем все кодовые слова
            prefix = word[:layer]  # префикс
            suffix = word[layer:]  # суффикс
            syndrome = partial_syndrome(
                prefix, columns[:layer], syndrome_length
            )  # частичный синдром
            nodes.setdefault(syndrome, set()).add(
                suffix
            )  # добавляем суффикс в множество

        layers.append(sorted(nodes))  # добавляем слой в список
        future_sets_by_layer.append(
            {syndrome: frozenset(nodes[syndrome]) for syndrome in nodes}
        )  # добавляем будущие множества в список

    return layers, future_sets_by_layer  # возвращаем списки слоев и будущих множеств


def build_future_trellis(codewords):  # строим решетку по множествам будущих
    n = len(codewords[0])  # длина кодового слова
    layers = []  # создаём список слоев

    for layer in range(n + 1):  # перебираем все слои
        nodes = {}  # создаём словарь узлов
        for word in codewords:  # перебираем все кодовые слова
            prefix = word[:layer]  # префикс
            suffix = word[layer:]  # суффикс
            nodes.setdefault(prefix, set()).add(suffix)  # добавляем суффикс в множество

        unique_future_sets = {}  # создаём словарь уникальных будущих множеств
        for prefix, suffixes in nodes.items():  # перебираем все узлы
            future = frozenset(suffixes)  # множество будущих
            unique_future_sets.setdefault(future, []).append(
                prefix
            )  # добавляем префикс в множество

        layers.append(unique_future_sets)  # добавляем слой в список

    return layers  # возвращаем список слоев


def compare_trellises(syndrome_future_sets, future_trellis):  # сравниваем решетки
    comparisons = []

    for layer_index, (
        syndrome_layer,
        future_layer,
    ) in enumerate(  # перебираем все слои и сравниваем множества
        zip(syndrome_future_sets, future_trellis)
    ):
        syndrome_signatures = sorted(
            syndrome_layer.values(), key=lambda item: sorted(item)
        )  # сортируем значения множеств
        future_signatures = sorted(
            future_layer.keys(), key=lambda item: sorted(item)
        )  # сортируем значения множеств
        comparisons.append(
            (layer_index, syndrome_signatures == future_signatures)
        )  # добавляем сравнение в список

    return comparisons  # возвращаем список сравнений

### Исходные данные и запуск

Строим `H`, перечисляем кодовые слова, затем выводим:

- узлы синдромной решетки;
- узлы решетки по множествам будущих;
- результат сравнения по ярусам.


In [17]:
G = [
    [1, 0, 1, 1, 0, 1],
    [1, 0, 1, 0, 1, 0],
    [1, 1, 0, 1, 0, 0],
]

codewords = all_codewords(G)  # перечисляем все кодовые слова
H = parity_check_matrix_from_generator(G)  # строим проверочную матрицу

print("Порождающая матрица кода:")
print_matrix("G", G)

print("\nПроверочная матрица:")
print_matrix("H", H)

print("\nПроверка G * H^T = 0:")
for row in G:
    products = [
        dot(row, check_row) for check_row in H
    ]  # перемножаем строку на проверочную матрицу
    print(" ", row, "->", products)

print("\nВсе кодовые слова:")
for index, word in enumerate(codewords):  # перебираем все кодовые слова
    print(f"{index}: {format_vector(word)}")

syndrome_layers, syndrome_future_sets = build_syndrome_trellis(
    codewords, H
)  # строим синдромную решетку
future_layers = build_future_trellis(codewords)  # строим решетку по множествам будущих
comparisons = compare_trellises(
    syndrome_future_sets, future_layers
)  # сравниваем решетки

print("\n" + "=" * 72)
print("Синдромная решетка")
print("=" * 72)
for layer_index, layer in enumerate(syndrome_layers):  # перебираем все слои
    formatted_nodes = [format_vector(node) for node in layer]  # форматируем узлы
    print(f"Ярус {layer_index}: {len(layer)} узл. -> {formatted_nodes}")  # выводим слой

print("\n" + "=" * 72)
print("Решетка по множествам будущих")
print("=" * 72)
for layer_index, layer in enumerate(future_layers):  # перебираем все слои
    print(f"Ярус {layer_index}: {len(layer)} узл.")  # выводим слой
    for node_index, (future_set, prefixes) in enumerate(
        layer.items(), start=1
    ):  # перебираем все узлы
        formatted_prefixes = [
            format_vector(prefix) for prefix in prefixes
        ]  # форматируем префиксы
        sample_future = sorted(
            format_vector(suffix) for suffix in future_set
        )  # форматируем будущие
        print(
            f"  Узел {node_index}: префиксы = {formatted_prefixes}, "
            f"число продолжений = {len(future_set)}, будущие = {sample_future}"
        )

print("\n" + "=" * 72)
print("Сравнение решеток")
print("=" * 72)
all_equal = True
for layer_index, equal in comparisons:
    print(f"Ярус {layer_index}: совпадение с точностью до нумерации узлов -> {equal}")
    all_equal = all_equal and equal

print("\nИтоговый профиль по ярусам:")
print("Синдромная решетка:", [len(layer) for layer in syndrome_layers])
print("Решетка по множествам будущих:", [len(layer) for layer in future_layers])

if all_equal:
    print(
        "\nВывод: построенная по проверочной матрице решетка совпадает "
        "с синдромной решеткой с точностью до нумерации узлов на каждом ярусе."
    )
else:
    print("\nВывод: решетки не совпали, требуется дополнительная проверка.")

Порождающая матрица кода:
G =
  [1, 0, 1, 1, 0, 1]
  [1, 0, 1, 0, 1, 0]
  [1, 1, 0, 1, 0, 0]

Проверочная матрица:
H =
  [0, 0, 1, 0, 1, 1]
  [0, 1, 0, 1, 0, 1]
  [1, 0, 0, 1, 1, 0]

Проверка G * H^T = 0:
  [1, 0, 1, 1, 0, 1] -> [0, 0, 0]
  [1, 0, 1, 0, 1, 0] -> [0, 0, 0]
  [1, 1, 0, 1, 0, 0] -> [0, 0, 0]

Все кодовые слова:
0: 000000
1: 110100
2: 101010
3: 011110
4: 101101
5: 011001
6: 000111
7: 110011

Синдромная решетка
Ярус 0: 1 узл. -> ['000']
Ярус 1: 2 узл. -> ['000', '001']
Ярус 2: 4 узл. -> ['000', '001', '010', '011']
Ярус 3: 4 узл. -> ['000', '011', '101', '110']
Ярус 4: 4 узл. -> ['000', '011', '101', '110']
Ярус 5: 2 узл. -> ['000', '110']
Ярус 6: 1 узл. -> ['000']

Решетка по множествам будущих
Ярус 0: 1 узл.
  Узел 1: префиксы = [''], число продолжений = 8, будущие = ['000000', '000111', '011001', '011110', '101010', '101101', '110011', '110100']
Ярус 1: 2 узл.
  Узел 1: префиксы = ['0'], число продолжений = 4, будущие = ['00000', '00111', '11001', '11110']
  Узел 2: преф